# Motion tracking

[Dataset](https://www.kaggle.com/datasets/smeschke/pedestrian-dataset?resource=download)

Use the following methods:
- Sparse Optical Flow
- Dense Optical Flow


In [1]:
import cv2 as cv
import numpy as np
import time
from imutils.object_detection import non_max_suppression

In [2]:
# Video file names
file1 = 'data/crosswalk.avi'
file2 = 'data/fourway.avi'
file3 = 'data/night.avi'

### Sparse optical flow

- Visualize trajectories of moving objects.
- Optional task: Identify each object and count them.
- Use following functions: `cv::goodFeaturesToTrack`, `cv::calcOpticalFlowPyrLK`


In [3]:
def sparse_optical_flow_v1(
        file: str,
        feature_params: dict,
        flow_params: dict,
        scale: float = 0.7,
        show_motion_mask: bool = False,
        show_features: bool = True,
):
    # Load the video
    cap = cv.VideoCapture(file)

    # Take the first frame of the video
    _, prev_frame = cap.read()
    # Convert the frame to grayscale
    prev_gray = cv.cvtColor(prev_frame, cv.COLOR_BGR2GRAY)

    # Get the initial feature points to track
    points0 = cv.goodFeaturesToTrack(prev_gray, mask=None, **feature_params)

    # Create some random colors
    colors = np.random.randint(0, 255, (len(points0), 3))

    # Create a mask image for drawing the trajectories
    motion_mask = np.zeros_like(prev_frame)

    while True:
        # Capture a frame from the video
        success, frame = cap.read()
        # End reading if there is no frame left
        if not success:
            cv.waitKey(0)
            break

        # Convert the frame to grayscale
        curr_gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

        # Calculate optical flow
        points1, status, errors = cv.calcOpticalFlowPyrLK(prev_gray, curr_gray, points0, None, **flow_params)

        # Select points that were successfully tracked
        old_points = points0[status == 1]
        new_points = points1[status == 1]

        # Draw the tracks
        for i, (new, old) in enumerate(zip(new_points, old_points)):
            a, b = new.ravel().astype(int)
            c, d = old.ravel().astype(int)
            color = colors[i].tolist()
            motion_mask = cv.line(motion_mask, (a, b), (c, d), color, 2)
            if show_features:
                frame = cv.circle(frame, (a, b), 5, color, -1)

        img = cv.add(frame, motion_mask)

        # Display the resulting image
        cv.imshow('Sparse Optical Flow', cv.resize(img, (0, 0), fx=scale, fy=scale))
        if show_motion_mask:
            cv.imshow('Motion Mask', cv.resize(motion_mask, (0, 0), fx=scale, fy=scale))

        # Wait for a key press
        if cv.waitKey(1) & 0xFF == ord('q'):
            break

        # Update the previous frame and points
        prev_gray = curr_gray.copy()
        points0 = new_points.reshape(-1, 1, 2)

    # Release the video capture and close all windows
    cap.release()
    cv.destroyAllWindows()


def sparse_optical_flow_v2(
        file: str,
        feature_params: dict,
        flow_params: dict,
        max_tracked_points=None,
        detecting_interval: int = 10,
        trajectory_length: int = 100,
        max_displacement_error: int = 50,
        scale: float = 0.7,
        show_features: bool = True,
):
    # Load the video
    cap = cv.VideoCapture(file)

    # Create some random colors
    num_colors = max_tracked_points if max_tracked_points is not None else 100
    colors = np.random.randint(0, 255, (num_colors, 3))

    # Take the first frame of the video and convert it to grayscale
    _, frame = cap.read()
    prev_gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    frame_idx = 0
    trajectories = []
    while True:
        start = time.time()

        # Capture a frame from the video
        success, frame = cap.read()

        # End reading if there is no frame
        if not success:
            cv.waitKey(0)
            break

        # Convert the frame to grayscale
        curr_gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

        # Detect new points to track if necessary
        if frame_idx % detecting_interval == 0 and \
                (max_tracked_points is None or len(trajectories) < max_tracked_points):
            mask = np.zeros_like(curr_gray)
            mask[:] = 255
            # Draw the latest points from each trajectory on the mask
            for p in [np.int32(trajectory[-1]) for trajectory in trajectories]:
                cv.circle(mask, p, 5, 0, -1)

            # Detect good features to track that are not already tracked using the mask
            pts = cv.goodFeaturesToTrack(curr_gray, mask=mask, **feature_params)
            if pts is not None:
                for p in np.float32(pts).reshape(-1, 2):
                    trajectories.append([p])

        if len(trajectories) > 0:
            # Extract the latest point from each trajectory
            pts0 = np.float32([trajectory[-1] for trajectory in trajectories]).reshape(-1, 1, 2)
            # Estimate the new positions of the tracked points in the current frame
            pts1, st, err = cv.calcOpticalFlowPyrLK(prev_gray, curr_gray, pts0, None, **flow_params)
            # Estimate the new positions of the tracked points in the previous frame (reverse optical flow)
            ptr0_rev, _, _ = cv.calcOpticalFlowPyrLK(curr_gray, prev_gray, pts1, None, **flow_params)
            # Calculate the displacement error between the original points and the reverse optical flow points
            displacement_error = abs(pts0 - ptr0_rev).reshape(-1, 2).max(-1)

            new_trajectories = []
            for i, ((x, y), trajectory) in enumerate(zip(pts1.reshape(-1, 2), trajectories)):
                # Skip the point if the displacement error is too high
                if displacement_error[i] > max_displacement_error:
                    continue

                # Add the new point to the trajectory
                trajectory.append((x, y))
                # Remove the oldest point if the trajectory is too long
                if trajectory_length is not None and len(trajectory) > trajectory_length:
                    del trajectory[0]

                # Draw the newest detected point on the image
                color = colors[i % num_colors].tolist()
                if show_features:
                    cv.circle(frame, (int(x), int(y)), 5, color, -1)

                # Keep only updated trajectories (with at least 2 points)
                if len(trajectory) > 1:
                    new_trajectories.append(trajectory)
                    # Draw the trajectory on the image
                    cv.polylines(frame, [np.int32(trajectory)], False, color, 2)
            trajectories = new_trajectories

        frame_idx += 1
        prev_gray = curr_gray

        # Calculate the FPS
        end = time.time()
        fps = 1 / (end - start)

        cv.putText(frame, f'Frame: {frame_idx}', (10, 30), cv.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv.LINE_AA)
        cv.putText(frame, f'FPS: {fps:.2f}', (10, 110), cv.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv.LINE_AA)
        cv.putText(frame, f'Tracked points: {len(trajectories)}', (10, 70), cv.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2,
                   cv.LINE_AA)
        cv.imshow('Sparse Optical Flow', cv.resize(frame, (0, 0), fx=scale, fy=scale))

        if cv.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv.destroyAllWindows()

In [69]:
# Define the paramseters for ShiTomasi corner detection
feature_params_1 = {
    'maxCorners': 200,
    'qualityLevel': 0.3,
    'minDistance': 7,
    'blockSize': 7,
}

# Define the parameters for Lucas Kanade optical flow
flow_params_1 = {
    'winSize': (15, 15),
    'maxLevel': 2,
    'criteria': (cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT, 10, 0.03),
}

sparse_optical_flow_v1(file1, feature_params_1, flow_params_1)
# sparse_optical_flow_v1(file2, feature_params_1, flow_params_1)
# sparse_optical_flow_v1(file3, feature_params_1, flow_params_1)

In [ ]:
feature_params_2 = {
    'maxCorners': 20,
    'qualityLevel': 0.3,
    'minDistance': 10,
    'blockSize': 7,
}

flow_params_2 = {
    'winSize': (15, 15),
    'maxLevel': 2,
    'criteria': (cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT, 10, 0.03),
}

sparse_optical_flow_v2(file1, feature_params_2, flow_params_2, 200)

In [ ]:
sparse_optical_flow_v2(file2, feature_params_2, flow_params_2, 200)

In [ ]:
sparse_optical_flow_v2(file3, feature_params_2, flow_params_2, 200)

### Dense optical flow

- Identify moving objects in video and draw green rectangle around them.
- Use downsampled video for this task if necessary for easier processing.
- Use following functions: `cv::calcOpticalFlowFarneback`


In [5]:
def dense_optical_flow(file, scale: float = 0.2, min_magnitude: float = 0.5, min_area: int = 100):
    # Load the video
    cap = cv.VideoCapture(file)

    # Read the first frame
    _, frame = cap.read()

    # Resize the first frame
    frame = cv.resize(frame, (0, 0), fx=scale, fy=scale)

    # Convert the first frame to grayscale
    prev_gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    # Create a mask image for drawing purposes
    hsv = np.zeros_like(frame)
    # Set the saturation to 255
    hsv[..., 1] = 255

    while True:
        # Read the next frame
        success, frame = cap.read()
        # Break if there are no more frames
        if not success:
            cv.waitKey(0)
            break

        # Resize the frame (for faster processing)
        frame = cv.resize(frame, (0, 0), fx=scale, fy=scale)

        # Convert the frame to grayscale
        curr_gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

        # Calculate the dense optical flow using Farneback's method
        flow = cv.calcOpticalFlowFarneback(prev_gray, curr_gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)

        # Calculate the magnitude and angle of the 2D vectors
        magnitude, angle = cv.cartToPolar(flow[..., 0], flow[..., 1], angleInDegrees=True)

        # Set image hue according to the optical flow direction
        hsv[..., 0] = angle / 2

        # Clip max value of the magnitude (because of possible infinity values)
        magnitude[magnitude > 10] = 10

        # Create a mask based on the magnitude
        mask = cv.inRange(magnitude, min_magnitude, 10)

        # Set image value according to the mask
        hsv[..., 2] = mask

        # Find the contours of the moving objects
        contours, _ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

        # Compute the bounding boxes of the contours
        rects = []
        for contour in contours:
            area = cv.contourArea(contour)
            if area > min_area:
                x, y, w, h = cv.boundingRect(contour)
                rects.append((x, y, x + w, y + h))
        rects = np.array(rects)

        # Remove overlapping bounding boxes
        rects = non_max_suppression(rects, probs=None, overlapThresh=0.2)

        # Draw the bounding boxes
        for x1, y1, x2, y2 in rects:
            cv.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Draw the frame with the optical flow vectors
        # Draw the optical flow
        cv.imshow('Video', frame)
        cv.imshow('Dense optical flow', cv.cvtColor(hsv, cv.COLOR_HSV2BGR))

        # Press 'q' on keyboard to exit
        if cv.waitKey(1) & 0xFF == ord('q'):
            break

        # Update the previous frame
        prev_gray = curr_gray

    # Release the video capture and video write objects
    cap.release()
    cv.destroyAllWindows()

In [8]:
dense_optical_flow(file1, scale=0.2, min_magnitude=0.4, min_area=10)

In [11]:
dense_optical_flow(file2, scale=0.2, min_magnitude=0.4, min_area=10)

In [19]:
dense_optical_flow(file3, scale=0.2, min_magnitude=0.4, min_area=200)

In [6]:
dense_optical_flow(0, scale=1.2, min_magnitude=5, min_area=20)